# Fuzzy Factory E-commerce Dataset 

This notebook was used to test out several data cleaning techniques that would eventually be implemented in the ETL pipeline to perform basic transformation.

## Extract data

In [ ]:
# Importing packages
import pandas as pd

import sys
sys.path.append("..")
from fuzzy import helpers

In [41]:
# Find all csv files in that folder
all_df = helpers.all_csv_to_df("../data/raw_data")


Importing:
products.csv: 4 rows, 3 columns
orders.csv: 32313 rows, 8 columns
website_sessions.csv: 472871 rows, 9 columns
order_item_refunds.csv: 1731 rows, 5 columns
order_items.csv: 40025 rows, 7 columns
website_pageviews.csv: 1188124 rows, 4 columns


In [42]:
# Get independent dataframes
refunds_df, order_item_df, orders_df, products_df, pageview_df, session_df = all_df["order_item_refunds.csv"],all_df["order_items.csv"], all_df["orders.csv"],all_df["products.csv"], all_df["website_pageviews.csv"], all_df["website_sessions.csv"]

## Missing values

In [43]:
# Check whether there are null values in the datasets
for name, df in all_df.items():
    if df.isna().any().any():
        print(f"{name} contains null values")
    else:
        print(f"0 null values in {name}")


0 null values in products.csv
0 null values in orders.csv
website_sessions.csv contains null values
0 null values in order_item_refunds.csv
0 null values in order_items.csv
0 null values in website_pageviews.csv


In [44]:
# Examine the null values in session_df
session_df[session_df.isna().any(axis=1)]

,website_session_id,created_at,user_id,is_repeat_session,utm_source,utm_campaign,utm_content,device_type,http_referer
904,905,2012-03-25 07:24:18,905,0,NaN,NaN,NaN,mobile,NaN
914,915,2012-03-25 10:45:03,915,0,NaN,NaN,NaN,mobile,NaN
933,934,2012-03-25 14:23:48,934,0,NaN,NaN,NaN,desktop,NaN
1044,1045,2012-03-26 11:54:49,846,1,NaN,NaN,NaN,desktop,NaN
1050,1051,2012-03-26 12:48:41,152,1,NaN,NaN,NaN,desktop,NaN
...,...,...,...,...,...,...,...,...,...
472835,472836,2015-03-19 06:25:38,394289,0,NaN,NaN,NaN,desktop,https://www.gsearch.com
472846,472847,2015-03-19 06:57:13,378178,1,NaN,NaN,NaN,mobile,https://www.gsearch.com
472860,472861,2015-03-19 07:38:28,383375,1,NaN,NaN,NaN,desktop,NaN
472863,472864,2015-03-19 07:41:11,394311,0,NaN,NaN,NaN,desktop,https://www.gsearch.com


In [45]:
# Replace null values with "unknown"
session_df = session_df.fillna("unknown")

## Extract datetime

In [46]:
# Convert string to datetime data type
session_df['created_at'] = pd.to_datetime(df['created_at'])

# Extract time and date components from the datetime variable
session_df['hour'] = session_df['created_at'].dt.hour
session_df['week_day'] = session_df['created_at'].dt.dayofweek
session_df['week_day_name'] = session_df['created_at'].dt.day_name()
session_df['day'] = session_df['created_at'].dt.day
session_df['month'] = session_df['created_at'].dt.month
session_df['month_name'] = session_df['created_at'].dt.month_name()
session_df['year'] = session_df['created_at'].dt.year

In [48]:
session_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 472871 entries, 0 to 472870
Data columns (total 16 columns):
 #   Column              Non-Null Count   Dtype         
---  ------              --------------   -----         
 0   website_session_id  472871 non-null  int64         
 1   created_at          472871 non-null  datetime64[us]
 2   user_id             472871 non-null  int64         
 3   is_repeat_session   472871 non-null  int64         
 4   utm_source          472871 non-null  str           
 5   utm_campaign        472871 non-null  str           
 6   utm_content         472871 non-null  str           
 7   device_type         472871 non-null  str           
 8   http_referer        472871 non-null  str           
 9   hour                472871 non-null  int32         
 10  week_day            472871 non-null  int32         
 11  week_day_name       472871 non-null  str           
 12  day                 472871 non-null  int32         
 13  month               472871 non-null  int

In [47]:
session_df

,website_session_id,created_at,user_id,is_repeat_session,utm_source,utm_campaign,utm_content,device_type,http_referer,hour,week_day,week_day_name,day,month,month_name,year
0,1,2012-03-19 08:04:16,1,0,gsearch,nonbrand,g_ad_1,mobile,https://www.gsearch.com,8,0,Monday,19,3,March,2012
1,2,2012-03-19 08:16:49,2,0,gsearch,nonbrand,g_ad_1,desktop,https://www.gsearch.com,8,0,Monday,19,3,March,2012
2,3,2012-03-19 08:26:55,3,0,gsearch,nonbrand,g_ad_1,desktop,https://www.gsearch.com,8,0,Monday,19,3,March,2012
3,4,2012-03-19 08:37:33,4,0,gsearch,nonbrand,g_ad_1,desktop,https://www.gsearch.com,8,0,Monday,19,3,March,2012
4,5,2012-03-19 09:00:55,5,0,gsearch,nonbrand,g_ad_1,mobile,https://www.gsearch.com,9,0,Monday,19,3,March,2012
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
472866,472867,2014-02-16 14:14:47,394314,0,gsearch,brand,g_ad_2,desktop,https://www.gsearch.com,14,6,Sunday,16,2,February,2014
472867,472868,2014-02-16 14:14:49,394315,0,bsearch,nonbrand,b_ad_1,mobile,https://www.bsearch.com,14,6,Sunday,16,2,February,2014
472868,472869,2014-02-16 14:16:05,394316,0,gsearch,nonbrand,g_ad_1,mobile,https://www.gsearch.com,14,6,Sunday,16,2,February,2014
472869,472870,2014-02-16 14:16:11,394317,0,gsearch,nonbrand,g_ad_1,desktop,https://www.gsearch.com,14,6,Sunday,16,2,February,2014
